In [ ]:
import pandas as pd
import os 
import sys
sys.path.append('../src')

from preprocessing import get_dfs, create_static_df, create_notes_df

In [ ]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)

In [ ]:
notes = create_notes_df(dfs)
notes

In [ ]:
print(dfs['exams'].columns)
print(dfs['exams'].count())

In [ ]:
print("Reading notes from exams.csv")
notes = dfs['exams'][['PatientID', 'TransplantationID', 'X', 'Befund', 'Art']].rename(
        columns={
            'PatientID': 'patient_id',
            'TransplantationID': 'transplant_id',
            'X': 'rel_days', 
            'Befund': 'text',
            'Art': 'type',
        }
    )
print(f"Found {len(notes)} texts")
notes

In [ ]:
print(dfs['clinical_assessment'].columns)
print(dfs['clinical_assessment'].count())

In [ ]:
# Select and rename the relevant columns from clinical_assessment to match the structure of notes
clinical_assessment_subset = dfs['clinical_assessment'][['PatientID', 'TransplantationID', 'OPDtime', 'BeurteilungAerztlich']].copy()
clinical_assessment_subset.rename(
    columns={
        'PatientID': 'patient_id',
        'TransplantationID': 'transplant_id',
        'OPDtime': 'rel_days',
        'BeurteilungAerztlich': 'text'
    },
    inplace=True
)

clinical_assessment_subset['type'] = 'clinical_assessment'
print(f"Loading {len(clinical_assessment_subset)} texts from clinical assessments")
# Append the transformed clinical_assessment data to notes
notes = pd.concat([notes, clinical_assessment_subset], ignore_index=True)
notes = notes.dropna(subset=['text'])

# remove patients not in static df
notes = notes.merge(
    static_df[['patient_id', 'transplant_id']],
    how='inner',  # Inner join to keep only matching entries
    on=['patient_id', 'transplant_id']
)

print(f"Concatenated texts and deleted NaNs, final count: {len(notes)}")
print(f"Average texts per patient: {len(notes) / len(static_df):.1f}")

In [ ]:
# Replace multiple <br> tags with a single whitespace in the notes
notes['text'] = notes['text'].str.replace(r'(<br>\s*)+', ' ', regex=True)

In [ ]:
notes

In [ ]:
notes